# DỰ ÁN CUỐI KHÓA "THE PRICE IS RIGHT"

Trong tuần này, chúng ta sẽ xây dựng một mô hình dự đoán giá của một sản phẩm dựa trên phần mô tả, sử dụng dữ liệu Amazon được thu thập bằng cách scrape.

# Trình tự thực hiện

NGÀY 1: Tuyển chọn dữ liệu  
NGÀY 2: Tiền xử lý dữ liệu  
NGÀY 3: Đánh giá, mô hình cơ sở và học máy truyền thống  
NGÀY 4: Học sâu và LLM  
NGÀY 5: Tinh chỉnh một mô hình nền tảng  

## NGÀY 3: Đánh giá, mô hình cơ sở và học máy truyền thống

Hôm nay, chúng ta sẽ viết một số mô hình đơn giản để dự đoán giá sản phẩm.

Chúng ta sẽ sử dụng một phương pháp để đánh giá hiệu quả của mô hình.

Sau đó, chúng ta sẽ thử nghiệm các mô hình cơ sở bằng học máy truyền thống.

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [ ]:
LITE_MODE = False

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Đã tải {len(train):,} mẫu huấn luyện, {len(val):,} mẫu kiểm định và {len(test):,} mẫu kiểm tra")

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

In [ ]:
# Khá thú vị!
# Chúng ta có thể làm tốt hơn với một mô hình đơn giản khác

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(f"Giá trung bình trong tập huấn luyện: {training_average}")

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# Hồi quy tuyến tính truyền thống!

np.random.seed(42)

# Tách các đặc trưng và biến mục tiêu
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Huấn luyện mô hình hồi quy tuyến tính
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Hệ số chặn: {model.intercept_}")

# Dự đoán trên tập kiểm tra và đánh giá
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Sai số bình phương trung bình: {mse}")
print(f"Điểm R bình phương: {r2}")

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
evaluate(linear_regression_pricer, test)

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [ ]:
# Đây là 1.000 từ phổ biến nhất được chọn, không bao gồm "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Số lượng từ được chọn: {len(selected_words)}")
print("Các từ được chọn:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

## Mô hình Random Forest

Random Forest là một dạng thuật toán **ensemble**, nghĩa là nó kết hợp nhiều thuật toán nhỏ hơn để đưa ra dự đoán tốt hơn.

Nó sử dụng một loại thuật toán học máy rất đơn giản gọi là **cây quyết định**. Cây quyết định đưa ra dự đoán bằng cách kiểm tra các giá trị đặc trưng đầu vào, tương tự một lưu đồ gồm các câu lệnh IF. Cây quyết định nhanh và đơn giản nhưng thường có xu hướng overfit.

Trong trường hợp này, "đặc trưng" là các phần tử của vector, nói cách khác là số lần một từ cụ thể xuất hiện trong phần mô tả sản phẩm.

Có thể hình dung như sau:

**Cây quyết định**  
\- NẾU từ "TV" xuất hiện nhiều hơn 3 lần THÌ  
-- NẾU từ "LED" xuất hiện nhiều hơn 2 lần THÌ  
--- NẾU từ "HD" xuất hiện ít nhất một lần THÌ  
---- Giá = 500 USD


Random Forest tạo ra nhiều cây quyết định. Mỗi cây được huấn luyện trên một tập con ngẫu nhiên khác nhau của dữ liệu và một tập con ngẫu nhiên khác nhau của các đặc trưng. Ở trên, chúng ta chỉ định 100 cây, đây cũng là giá trị mặc định.

Sau đó, mô hình Random Forest lấy giá trị trung bình dự đoán từ tất cả các cây để tạo ra kết quả cuối cùng.

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

In [ ]:
# Đây là cách lưu mô hình nếu bạn cần, đặc biệt khi chạy trên một tập dữ liệu lớn hơn

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Giới thiệu XGBoost

Giống Random Forest, XGBoost cũng là một mô hình ensemble kết hợp nhiều cây quyết định.

Tuy nhiên, khác với Random Forest, XGBoost xây dựng từng cây nối tiếp nhau; mỗi cây tiếp theo sẽ sửa các lỗi của cây trước đó bằng phương pháp "gradient descent".

XGBoost nhanh hơn nhiều so với Random Forest, vì vậy chúng ta có thể chạy trên toàn bộ tập dữ liệu. Mô hình này thường có khả năng tổng quát hóa tốt hơn.

**Nếu lệnh import này không hoạt động, bạn có thể bỏ qua phần này vì đây không phải nội dung bắt buộc. Trên máy Mac, có thể cần chạy `brew install libomp` trong terminal.**

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng trong kinh doanh</h2>
            <span style="color:#181;">Học máy truyền thống không chỉ hữu ích để tìm hiểu lịch sử; ngày nay nó vẫn được sử dụng rộng rãi trong ngành, đặc biệt với các bài toán có những đặc trưng nhận diện rõ ràng. Hãy dành thời gian khám phá các thuật toán và thử nghiệm. Biết đâu bạn có thể vượt qua các kết quả của tôi bằng học máy truyền thống! Tôi đã chạy Random Forest trên toàn bộ 800.000 mẫu huấn luyện. Quá trình này mất khoảng 15 giờ và đạt sai số thấp ở mức 56,40 USD. Học máy truyền thống có thể cho kết quả tốt, hãy tự mình thử nghiệm.</span>
        </td>
    </tr>
</table>